In [1]:
import pandas as pd
from datetime import datetime
import random

In [6]:
customers = pd.DataFrame({
    "customer_id": range(1, 9),
    "name": [
        "Ali Mammadov", "Nigar Aliyeva", "Rashad Karimov", "Aysel Hasanli",
        "Orkhan Guliyev", "Leyla Abbasova", "Tural Ismayilov", "Gunel Rahimova"
    ],
    "email": [
        "ali@mail.com", "nigar@mail.com", "rashad@mail.com", "aysel@mail.com",
        "orkhan@mail.com", "leyla@mail.com", "tural@mail.com", "gunel@mail.com"
    ],
    "city": [
        "Baku", "Ganja", "Baku", "Sumqayit",
        "Baku", "Ganja", "Lankaran", "Baku"
    ],
    "signup_date": pd.date_range(start="2023-01-01", periods=8, freq='W')
})


movies = pd.DataFrame({
    "movie_id": range(1, 11),
    "title": [
        "Inception", "Titanic", "Interstellar", "The Dark Knight",
        "Avengers", "Gladiator", "The Matrix", "Joker",
        "Parasite", "The Godfather"
    ],
    "genre": [
        "Sci-Fi", "Romance", "Sci-Fi", "Action",
        "Action", "Drama", "Sci-Fi", "Drama",
        "Thriller", "Crime"
    ],
    "release_year": [
        2010, 1997, 2014, 2008,
        2012, 2000, 1999, 2019,
        2019, 1972
    ],
    "rating": [5, 4, 5, 5, 4, 4, 5, 4, 5, 5]
})


rentals_list = []

for rental_id in range(1, 26):
    customer_id = random.randint(1, 8)
    movie_id = random.randint(1, 10)
    rental_date = datetime(2024, random.randint(1, 12), random.randint(1, 28))
    return_date = rental_date
    
    price = round(random.uniform(3.99, 9.99), 2)

    rentals_list.append([
        rental_id,
        customer_id,
        movie_id,
        rental_date,
        return_date,
        price
    ])

rentals = pd.DataFrame(rentals_list, columns=[
    "rental_id",
    "customer_id",
    "movie_id",
    "rental_date",
    "return_date",
    "price"
])

### 1. Find all movies rented by a specific customer (requires joining rentals with movies).

In [9]:
customer_id = 1 

customer_rentals = rentals[rentals["customer_id"] == customer_id]

result = customer_rentals.merge(movies, on="movie_id")

print(result[["title", "genre", "rental_date", "price"]])

             title    genre rental_date  price
0          Titanic  Romance  2024-11-27   9.15
1            Joker    Drama  2024-04-16   7.56
2         Avengers   Action  2024-06-01   6.61
3  The Dark Knight   Action  2024-06-10   6.33


### 2. Find the top 5 most-rented movies (group and count).

In [11]:
top_movies = (
    rentals
    .groupby("movie_id")
    .size()
    .reset_index(name="rental_count")
    .merge(movies, on="movie_id")
    .sort_values("rental_count", ascending=False)
    .head(5)
)

print(top_movies[["title", "rental_count"]])

           title  rental_count
7          Joker             6
0      Inception             5
1        Titanic             3
8  The Godfather             3
2   Interstellar             2


### 3. Compute the total revenue per genre (join rentals with movies, group by genre, sum price).

In [12]:
revenue_by_genre = (
    rentals
    .merge(movies, on="movie_id")
    .groupby("genre")["price"]
    .sum()
    .reset_index()
)

print(revenue_by_genre)

     genre  price
0   Action  17.36
1    Crime  21.55
2    Drama  60.43
3  Romance  27.00
4   Sci-Fi  59.13


### 4. Find customers who have never rented a movie rated below 3 (multi-step filtering).

In [13]:

low_rated_movies = movies[movies["rating"] < 3]["movie_id"]

customers_with_low_rentals = rentals[
    rentals["movie_id"].isin(low_rated_movies)
]["customer_id"].unique()


safe_customers = customers[
    ~customers["customer_id"].isin(customers_with_low_rentals)
]

print(safe_customers[["name", "email"]])

              name            email
0     Ali Mammadov     ali@mail.com
1    Nigar Aliyeva   nigar@mail.com
2   Rashad Karimov  rashad@mail.com
3    Aysel Hasanli   aysel@mail.com
4   Orkhan Guliyev  orkhan@mail.com
5   Leyla Abbasova   leyla@mail.com
6  Tural Ismayilov   tural@mail.com
7   Gunel Rahimova   gunel@mail.com


In [14]:
# Step 3

denormalized = (
    rentals
    .merge(customers, on="customer_id")
    .merge(movies, on="movie_id")
)

denormalized.head()

,rental_id,customer_id,movie_id,rental_date,return_date,price,name,email,city,signup_date,title,genre,release_year,rating
0,1,2,8,2024-01-27,2024-01-27,8.53,Nigar Aliyeva,nigar@mail.com,Ganja,2023-01-08,Joker,Drama,2019,4
1,2,8,1,2024-12-02,2024-12-02,5.53,Gunel Rahimova,gunel@mail.com,Baku,2023-02-19,Inception,Sci-Fi,2010,5
2,3,2,2,2024-01-14,2024-01-14,8.44,Nigar Aliyeva,nigar@mail.com,Ganja,2023-01-08,Titanic,Romance,1997,4
3,4,6,1,2024-01-16,2024-01-16,9.95,Leyla Abbasova,leyla@mail.com,Ganja,2023-02-05,Inception,Sci-Fi,2010,5
4,5,5,8,2024-09-15,2024-09-15,8.37,Orkhan Guliyev,orkhan@mail.com,Baku,2023-01-29,Joker,Drama,2019,4


## Normalization vs Denormalization

When we join customers, rentals, and movies into a single denormalized table, significant redundancy appears.

Redundancy examples:
- Customer information (name, email, city) is repeated for every rental made by the same customer.
- Movie information (title, genre, release_year, rating) is repeated for every rental of that movie.

This causes several problems:
1. Storage inefficiency – repeated text data increases size.
2. Update anomalies – if a customer updates their email, every row must be updated.
3. Risk of inconsistency – if one row is missed during update, the database becomes inconsistent.

In the normalized schema:
- Customer data is stored only once in the `customers` table.
- Movie data is stored only once in the `movies` table.
- The `rentals` table only stores foreign keys (customer_id, movie_id).

This eliminates redundancy, ensures consistency, and makes updates safe and efficient.

In [15]:
customer_documents = []

for _, customer in customers.iterrows():
    
    # Bu müştərinin bütün rental-ları
    customer_rentals = rentals[rentals["customer_id"] == customer["customer_id"]]
    
    rental_list = []
    
    for _, rental in customer_rentals.iterrows():
        
        # Film məlumatını tapırıq
        movie = movies[movies["movie_id"] == rental["movie_id"]].iloc[0]
        
        rental_entry = {
            "rental_id": rental["rental_id"],
            "rental_date": str(rental["rental_date"]),
            "return_date": str(rental["return_date"]),
            "price": rental["price"],
            "movie": {
                "movie_id": movie["movie_id"],
                "title": movie["title"],
                "genre": movie["genre"],
                "release_year": movie["release_year"],
                "rating": movie["rating"]
            }
        }
        
        rental_list.append(rental_entry)
    
    customer_document = {
        "customer_id": customer["customer_id"],
        "name": customer["name"],
        "email": customer["email"],
        "city": customer["city"],
        "signup_date": str(customer["signup_date"]),
        "rentals": rental_list
    }
    
    customer_documents.append(customer_document)

customer_documents[:2]  

[{'customer_id': 1,
  'name': 'Ali Mammadov',
  'email': 'ali@mail.com',
  'city': 'Baku',
  'signup_date': '2023-01-01 00:00:00',
  'rentals': [{'rental_id': 7,
    'rental_date': '2024-11-27 00:00:00',
    'return_date': '2024-11-27 00:00:00',
    'price': 9.15,
    'movie': {'movie_id': np.int64(2),
     'title': 'Titanic',
     'genre': 'Romance',
     'release_year': np.int64(1997),
     'rating': np.int64(4)}},
   {'rental_id': 13,
    'rental_date': '2024-04-16 00:00:00',
    'return_date': '2024-04-16 00:00:00',
    'price': 7.56,
    'movie': {'movie_id': np.int64(8),
     'title': 'Joker',
     'genre': 'Drama',
     'release_year': np.int64(2019),
     'rating': np.int64(4)}},
   {'rental_id': 16,
    'rental_date': '2024-06-01 00:00:00',
    'return_date': '2024-06-01 00:00:00',
    'price': 6.61,
    'movie': {'movie_id': np.int64(5),
     'title': 'Avengers',
     'genre': 'Action',
     'release_year': np.int64(2012),
     'rating': np.int64(4)}},
   {'rental_id': 23,
  

## **Step 2:** Execute the same four queries from Task 1 on the document collection:
### 1. Find all movies rented by a specific customer.

In [16]:
customer_id = 1

for customer in customer_documents:
    if customer["customer_id"] == customer_id:
        rented_movies = [r["movie"]["title"] for r in customer["rentals"]]
        print(rented_movies)
        break

['Titanic', 'Joker', 'Avengers', 'The Dark Knight']


### 2. Find the top 5 most-rented movies.

In [17]:
from collections import Counter

movie_counter = Counter()

for customer in customer_documents:
    for rental in customer["rentals"]:
        movie_title = rental["movie"]["title"]
        movie_counter[movie_title] += 1

top_5 = movie_counter.most_common(5)
print(top_5)

[('Joker', 6), ('Inception', 5), ('Titanic', 3), ('The Godfather', 3), ('Avengers', 2)]


### 3. Compute total revenue per genre.

In [18]:
from collections import defaultdict

revenue_by_genre = defaultdict(float)

for customer in customer_documents:
    for rental in customer["rentals"]:
        genre = rental["movie"]["genre"]
        revenue_by_genre[genre] += rental["price"]

print(dict(revenue_by_genre))

{'Romance': 27.0, 'Drama': 60.42999999999999, 'Action': 17.36, 'Sci-Fi': 59.13, 'Crime': 21.549999999999997}


### 4. Find customers who have never rented a movie rated below 3.

In [19]:
safe_customers = []

for customer in customer_documents:
    
    rented_low_rated = False
    
    for rental in customer["rentals"]:
        if rental["movie"]["rating"] < 3:
            rented_low_rated = True
            break
    
    if not rented_low_rated:
        safe_customers.append(customer["name"])

print(safe_customers)

['Ali Mammadov', 'Nigar Aliyeva', 'Rashad Karimov', 'Aysel Hasanli', 'Orkhan Guliyev', 'Leyla Abbasova', 'Tural Ismayilov', 'Gunel Rahimova']


## Document vs Relational Queries

**Easier in document model:**  
- Find all movies rented by a specific customer.  
- Rentals are embedded in the customer document, so no joins are needed.

**Harder in document model:**  
- Top 5 most-rented movies  
- Total revenue per genre  
- Customers who never rented a movie rated below 3  
- Requires looping through all documents and nested lists; relational `groupby` and `merge` are simpler.

**Document model advantage:**  
- Fast, simple access to a single entity and its related data.  
- Great for entity-centric queries (e.g., show full customer profile).

**Relational model advantage:**  
- Efficient for cross-entity aggregation, analytics, and complex filtering.  
- Better for reporting and ensuring consistency.

In [20]:
nodes = {
    "c001": {"type": "customer", "name": "Alice"},
    "m001": {"type": "movie", "title": "Inception"},
    "action": {"type": "genre", "name": "Action"},
    "nyc": {"type": "city", "name": "New York"}
}

edges = [
    ("c001", "rented", "m001"),     
    ("m001", "belongs_to", "action"), 
    ("c001", "lives_in", "nyc")       
]

alice_rentals = [tgt for src, rel, tgt in edges if src == "c001" and rel == "rented"]
print(alice_rentals)

action_movies = [src for src, rel, tgt in edges if rel == "belongs_to" and tgt == "action"]
customers = [src for src, rel, tgt in edges if rel == "rented" and tgt in action_movies]
print(customers)

['m001']
['c001']


1. "Which movies did customer X rent?" — one hop from customer through `rented` edges.
2. "Which genres does customer X prefer?" — two hops: customer → movies → genres, then count.
3. "Which customers share a genre preference with customer X?" — three hops: customer X → movies → genres → other customers who rented movies in the same genre.
4. "Recommend movies for customer X" — find movies in their preferred genres that they have not yet rented.

### 1. "Which movies did customer X rent?" — one hop from customer through `rented` edges.

In [21]:
def movies_rented_by(customer_id, edges):
    return [tgt for src, rel, tgt in edges if src == customer_id and rel == "rented"]

# Example
print("Movies rented by Alice:", movies_rented_by("c001", edges))

Movies rented by Alice: ['m001']


### 2. "Which genres does customer X prefer?" — two hops: customer → movies → genres, then count.


In [22]:
from collections import Counter

def genres_preferred_by(customer_id, edges):
    rented_movies = movies_rented_by(customer_id, edges)  
    genres = [tgt for src, rel, tgt in edges if src in rented_movies and rel == "belongs_to"]  
    return dict(Counter(genres))  


print("Genres preferred by Alice:", genres_preferred_by("c001", edges))

Genres preferred by Alice: {'action': 1}


### 3. "Which customers share a genre preference with customer X?" — three hops: customer X → movies → genres → other customers who rented movies in the same genre.


In [23]:
def customers_with_shared_genres(customer_id, edges):
    preferred_genres = genres_preferred_by(customer_id, edges).keys()  # 2nd hop
    
    movies_in_genres = [src for src, rel, tgt in edges if rel == "belongs_to" and tgt in preferred_genres]
    
    other_customers = set(src for src, rel, tgt in edges if rel == "rented" and tgt in movies_in_genres)
    other_customers.discard(customer_id)  
    return list(other_customers)


print("Customers who share a genre with Alice:", customers_with_shared_genres("c001", edges))

Customers who share a genre with Alice: []


### 4. "Recommend movies for customer X" — find movies in their preferred genres that they have not yet rented.

In [24]:
def recommend_movies(customer_id, edges):
    preferred_genres = genres_preferred_by(customer_id, edges).keys()
    rented_movies = set(movies_rented_by(customer_id, edges))
    
    movies_in_genres = set(src for src, rel, tgt in edges if rel == "belongs_to" and tgt in preferred_genres)
    
    recommendations = movies_in_genres - rented_movies
    return list(recommendations)


print("Recommended movies for Alice:", recommend_movies("c001", edges))

Recommended movies for Alice: []


### Why Graph Traversal is Natural for Recommendations

The recommendation query involves multiple hops:

1. Start from a customer → find the movies they rented.  
2. From those movies → find the genres.  
3. From those genres → find all other movies in the same genres.  
4. Exclude movies the customer has already rented.  

In **SQL**, this requires multiple joins and subqueries, and you must know in advance the number of relationship levels. Variable-length paths are cumbersome or sometimes impossible to express.  

In a **document model**, the data is nested by entity, not by relationships. Answering this query would require scanning all documents, extracting genres from each rental, and recombining them, which is inefficient.  

The **graph model** is natural because:  

- Relationships are first-class citizens (edges).  
- Traversals of arbitrary length are straightforward.  
- Counting or filtering along paths is easy.  
- Recommendations are just a few hops away from the starting node.  

 **Takeaway:** Graphs excel at queries where **connections matter more than individual entities**, such as recommendations, social networks, and fraud detection.

In [27]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random


np.random.seed(42)
random.seed(42)


transaction_ids = [f"T{str(i).zfill(5)}" for i in range(1, 5001)]


customer_ids = [f"C{random.randint(1, 500)}" for _ in range(5000)]


product_ids = [f"P{random.randint(1, 100)}" for _ in range(5000)]


amounts = np.round(np.random.uniform(5, 500, 5000), 2)


payment_methods = np.random.choice(["Credit Card", "PayPal", "Bank Transfer"], size=5000)

start_date = datetime(2025, 1, 1)
timestamps = [start_date + timedelta(days=random.randint(0, 365),
                                     hours=random.randint(0, 23),
                                     minutes=random.randint(0, 59)) for _ in range(5000)]


statuses = np.random.choice(["completed", "refunded", "pending"], size=5000, p=[0.8, 0.1, 0.1])


transactions_df = pd.DataFrame({
    "transaction_id": transaction_ids,
    "customer_id": customer_ids,
    "product_id": product_ids,
    "amount": amounts,
    "payment_method": payment_methods,
    "timestamp": timestamps,
    "status": statuses
})

transactions_df.head()

,transaction_id,customer_id,product_id,amount,payment_method,timestamp,status
0,T00001,C328,P28,190.40,Bank Transfer,2025-05-04 02:35:00,refunded
1,T00002,C58,P62,475.60,Bank Transfer,2025-01-07 22:46:00,refunded
2,T00003,C13,P10,367.34,Bank Transfer,2025-07-28 06:08:00,pending
3,T00004,C380,P17,301.34,PayPal,2025-07-03 09:15:00,completed
4,T00005,C141,P76,82.23,Bank Transfer,2025-11-28 02:01:00,completed


In [28]:

transaction_id_to_lookup = "T0001"
transaction = transactions_df[transactions_df["transaction_id"] == transaction_id_to_lookup]
print("Lookup transaction:")
print(transaction)


new_transaction = {
    "transaction_id": "T5001",
    "customer_id": "C501",
    "product_id": "P101",
    "amount": 120.50,
    "payment_method": "Credit Card",
    "timestamp": datetime(2025, 3, 4, 14, 30),
    "status": "completed"
}

transactions_df = pd.concat([transactions_df, pd.DataFrame([new_transaction])], ignore_index=True)
print("\nAfter inserting new transaction, last row:")
print(transactions_df.tail(1))


pending_mask = transactions_df["status"] == "pending"
transactions_df.loc[pending_mask, "status"] = "completed"
print(f"\nNumber of transactions updated from pending to completed: {pending_mask.sum()}")


valid_statuses = ["completed", "refunded", "pending"]
invalid_transactions = transactions_df[
    (transactions_df["amount"] <= 0) | (~transactions_df["status"].isin(valid_statuses))
]

print("\nInvalid transactions (if any):")
print(invalid_transactions)

Lookup transaction:
Empty DataFrame
Columns: [transaction_id, customer_id, product_id, amount, payment_method, timestamp, status]
Index: []

After inserting new transaction, last row:
     transaction_id customer_id product_id  amount payment_method  \
5000          T5001        C501       P101   120.5    Credit Card   

               timestamp     status  
5000 2025-03-04 14:30:00  completed  

Number of transactions updated from pending to completed: 481

Invalid transactions (if any):
Empty DataFrame
Columns: [transaction_id, customer_id, product_id, amount, payment_method, timestamp, status]
Index: []



### OLTP vs OLAP Storage Layout Explanation
**OLTP (Online Transaction Processing)** operations usually work with **individual records** — inserting, updating, or reading a single transaction at a time. Row-major storage stores **all fields of a record together** in memory, so accessing or updating a full record is fast and efficient. This layout matches OLTP’s **row-level access pattern** perfectly.

**OLAP (Online Analytical Processing)** operations, on the other hand, often **scan or aggregate a single column across thousands or millions of rows** — for example, computing the total sales or average amount. Column-major storage stores **all values of a column contiguously**, allowing these column-wise operations to be performed as a fast sequential read. This reduces memory fetches and takes advantage of CPU caching, making analytics queries much faster.

**Key idea:**  
- **OLTP → row-major → fast full-record access**  
- **OLAP → column-major → fast column aggregation**

This alignment between workload patterns and memory layout is why OLTP databases prioritize row storage while OLAP databases optimize for column storage.

### Task 5: Recommended Data Models and Processing Paradigms

1. **Hospital Patient Record System**  
   - **Data model:** Relational  
   - **Processing paradigm:** OLTP  
   - **Justification:** Patient records have structured, interrelated data (appointments, diagnoses, treatments). Doctors need consistent, low-latency access to complete records, which fits **row-major transactional processing** and relational tables enforcing ACID guarantees.  

2. **Social Network (Friends of Friends)**  
   - **Data model:** Graph  
   - **Processing paradigm:** OLTP / traversal-focused  
   - **Justification:** Relationships between users are central. Graph databases excel at multi-hop queries like "friends of friends," which are complex to express in SQL. Fast traversal operations make real-time recommendations feasible.  

3. **Financial Reporting System**  
   - **Data model:** Relational  
   - **Processing paradigm:** OLAP  
   - **Justification:** Structured, tabular data across products, regions, and time periods is ideal for aggregation queries. Column-major storage and batch processing allow fast computation of revenue breakdowns over millions of transactions.  

4. **IoT Platform (Real-time Sensor Data)**  
   - **Data model:** Document or Time-series optimized storage  
   - **Processing paradigm:** OLTP / near real-time streaming  
   - **Justification:** Each sensor may emit slightly different fields or metadata, making a flexible schema (document) practical. OLTP-style ingestion with low-latency writes ensures anomalies are detected promptly.  

5. **Content Management System (Variable Article Metadata)**  
   - **Data model:** Document  
   - **Processing paradigm:** OLTP / mixed queries  
   - **Justification:** Articles have varying fields (tags, authors, images, custom metadata). Document databases handle schema-on-read naturally, allowing flexible storage and fast retrieval of individual articles without enforcing a rigid schema.